# CC-ResDiff on Colab
### Global Color-Consistency Supervision for Residual-Space Diffusion Super-Resolution

Reproduces the CelebA experiment end to end: **baseline SRDiff** vs **SRDiff + the Global Color-Consistency (GCC) loss**, at a reduced scale (16x16 -> 64x64, 4x) that fits a single T4.

**Run order:** 1 setup -> 2 install -> 3 data -> 4 Stage 1 (RRDB) -> 5 Stage 2 (both arms) -> 6 evaluate -> 7 compare. Sections 8-10 are optional (gradient diagnostic, ablations, DIV2K).

**Approximate T4 runtimes:** Stage 1 ~30-40 min, each Stage 2 arm ~40-60 min, each evaluation ~5-10 min. Budget ~2.5 hours for sections 4-7.

**Set Runtime -> Change runtime type -> T4 GPU before starting.**

Everything is written into a Google Drive folder, so a disconnect does not lose work: re-running a training cell resumes from the last checkpoint automatically.

## 1. Setup: GPU, Drive, code

In [1]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
import multiprocessing
print('cpu cores:', multiprocessing.cpu_count())

Thu Aug  6 15:18:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/SUDHIR-KUMAR-06/-Global-Color-Consistency-Supervision-for-Residual-Space-Diffusion-Super-Resolution.git'
PROJECT_DIR = '/content/drive/MyDrive/CC-ResDiff'   # checkpoints + data live here, so they survive disconnects

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
REPO_DIR = f'{PROJECT_DIR}/SRDiff'

if not os.path.exists(REPO_DIR):
    !git clone "{REPO_URL}" "{REPO_DIR}"
else:
    print('repo present, pulling latest')
    !cd "{REPO_DIR}" && git pull --ff-only

%cd {REPO_DIR}
!git log --oneline -1
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo present, pulling latest
Already up to date.
/content/drive/MyDrive/CC-ResDiff/SRDiff
f0e8f60 (HEAD -> main, origin/main, origin/HEAD) Rewrite the Colab notebook against the current code
 CC_ResDiff_colab.ipynb   data_gen	     models	        tasks
 configs		 'kaggle (1).json'   readme.md	        utils
 data			  kaggle.json	     requirements.txt


## 2. Install dependencies

Colab already ships torch/torchvision; the rest come from `requirements.txt`. Restart the runtime only if pip reports a conflict it could not resolve.

In [8]:
!pip install -q lpips pytorch-fid einops tensorboardX PyYAML scikit-image opencv-python natsort
import lpips, pytorch_fid, skimage, einops
print('imports OK')

imports OK


## 3. CelebA data

The packer needs, relative to the repo root:

```
data/raw/celebA/img_align_celeba/img_align_celeba/*.jpg
data/raw/celebA/list_eval_partition.csv
```

Both the Kaggle `.csv` and the original space-separated `.txt` partition formats are auto-detected. If your copy sits elsewhere, point `celeba_img_dir` / `celeba_partition_file` at it (see `configs/celeb_a.yaml`) instead of moving files.

**Option A (recommended): Kaggle API.** Fast and reliable. Needs `kaggle.json` from your Kaggle account (Settings -> API -> Create New Token).

**Option B: copy from Drive** if you already uploaded the dataset there.

`torchvision.datasets.CelebA(download=True)` is deliberately *not* used here: it pulls from a Google Drive link that is frequently quota-blocked.

In [9]:
# --- Option A: Kaggle API ---
# Upload kaggle.json when prompted.
import os

RAW = 'data/raw/celebA'
if not os.path.exists(f'{RAW}/list_eval_partition.csv'):
    from google.colab import files
    print('Upload your kaggle.json')
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle
    !mkdir -p {RAW}
    !kaggle datasets download -d jessicali9530/celeba-dataset -p {RAW} --unzip
else:
    print('CelebA already present')

!ls {RAW}
!ls {RAW}/img_align_celeba | head -3

Upload your kaggle.json


Saving kaggle.json to kaggle (2).json
Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
celeba-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
celeba-dataset.zip  img_align_celeba
img_align_celeba


In [5]:
# --- Option B: copy from Drive (skip if Option A worked) ---
# SRC = '/content/drive/MyDrive/celebA'   # your uploaded copy
# !mkdir -p data/raw && cp -r "{SRC}" data/raw/celebA
# !ls data/raw/celebA

In [10]:
# Verify the layout the packer expects, before spending time on packing.
import os, glob

img_dir = 'data/raw/celebA/img_align_celeba/img_align_celeba'
part = 'data/raw/celebA/list_eval_partition.csv'
if not os.path.isdir(img_dir):
    alt = 'data/raw/celebA/img_align_celeba'
    if os.path.isdir(alt) and glob.glob(f'{alt}/*.jpg'):
        img_dir = alt
if not os.path.exists(part):
    for cand in ['data/raw/celebA/list_eval_partition.txt',
                 'data/raw/celebA/Eval/list_eval_partition.txt']:
        if os.path.exists(cand):
            part = cand
            break
n = len(glob.glob(f'{img_dir}/*.jpg'))
print('image dir :', img_dir, '->', n, 'jpgs')
print('partition :', part, '->', os.path.exists(part))
assert n > 0, 'no jpgs found - check the CelebA download/copy cell above'
assert os.path.exists(part), 'no partition file found'

# Only needed if your paths differ from the config defaults.
OVERRIDE = f'celeba_img_dir={img_dir},celeba_partition_file={part}'
print('\npacker override string:\n  --hparams="' + OVERRIDE + '"')

image dir : data/raw/celebA/img_align_celeba/img_align_celeba -> 49523 jpgs
partition : data/raw/celebA/list_eval_partition.csv -> False


AssertionError: no partition file found

In [ ]:
# Pack a subsampled binary dataset: 3000 train / 300 valid / 300 test at 64x64 HR (16x16 LR).
# Sizes come from configs/celeb_a_small.yaml (max_train_imgs etc). Takes ~1 min.
# OVERRIDE comes from the verification cell above and handles non-default layouts.
!PYTHONPATH=. python data_gen/celeb_a.py --config configs/celeb_a_small.yaml --hparams="{OVERRIDE}"
!du -sh data/binary/celeb_a_sr

## 4. Stage 1: RRDB conditioning net

Unmodified SRDiff — CC-ResDiff only changes Stage 2. Both diffusion arms share this checkpoint, so it is trained once.

`NUM_WORKERS` is set from the actual core count: Colab usually gives 2 vCPUs, and the repo default of 6 would oversubscribe them.

In [ ]:
import multiprocessing
NUM_WORKERS = max(1, min(4, multiprocessing.cpu_count() - 1))
print('using num_workers =', NUM_WORKERS)

In [ ]:
!PYTHONPATH=. python -u tasks/trainer.py \
  --config configs/rrdb/celeb_a_pretrain_small.yaml \
  --exp_name rrdb_celebA_small --reset \
  --hparams="num_workers={NUM_WORKERS}"

## 5. Stage 2: diffusion, baseline vs CC-ResDiff

The two configs are identical except for `use_color_loss` (and `lambda_color` / `color_pool_size`), so any difference in results is attributable to the GCC loss.

Both use early stopping on **PSNR** — deliberately not on `color_error`, since stopping on the metric the method targets would bias the comparison toward CC-ResDiff. The evaluation later loads `model_ckpt_best.ckpt`, not the final step.

Re-run either cell after a disconnect and it resumes from its last checkpoint.

In [ ]:
# Arm 1: baseline (no GCC loss)
!PYTHONPATH=. python -u tasks/trainer.py \
  --config configs/diffsr_celeb_small.yaml \
  --exp_name diffsr_celebA_small_baseline --reset \
  --hparams="rrdb_ckpt=checkpoints/rrdb_celebA_small,num_workers={NUM_WORKERS}"

In [ ]:
# Arm 2: CC-ResDiff (GCC loss, lambda_color=0.1, color_pool_size=8)
!PYTHONPATH=. python -u tasks/trainer.py \
  --config configs/diffsr_celeb_small_cc.yaml \
  --exp_name diffsr_celebA_small_cc --reset \
  --hparams="rrdb_ckpt=checkpoints/rrdb_celebA_small,num_workers={NUM_WORKERS}"

## 6. Evaluate both arms

Runs the 300-image test split and reports PSNR / SSIM / LPIPS / LR-PSNR / **color_error** plus **FID**, writing `metrics.json` under `checkpoints/<exp>/results_<step>_/`.

Both arms are evaluated under the same seed, so they see identical sampling noise — the comparison is paired.

In [ ]:
!PYTHONPATH=. python -u tasks/evaluate.py \
  --config configs/diffsr_celeb_small.yaml --exp_name diffsr_celebA_small_baseline

In [ ]:
!PYTHONPATH=. python -u tasks/evaluate.py \
  --config configs/diffsr_celeb_small_cc.yaml --exp_name diffsr_celebA_small_cc

## 7. Compare

In [ ]:
import glob, json
import pandas as pd


def latest_metrics(exp):
    paths = sorted(glob.glob(f'checkpoints/{exp}/results_*/metrics.json'))
    assert paths, f'no metrics for {exp} - run its evaluation cell first'
    with open(paths[-1]) as f:
        return json.load(f)


base = latest_metrics('diffsr_celebA_small_baseline')
cc = latest_metrics('diffsr_celebA_small_cc')
df = pd.DataFrame({'SRDiff (baseline)': base, 'SRDiff + CC-ResDiff': cc}).T

# lower is better for these
LOWER = {'lpips', 'color_error', 'fid'}
delta = {}
for k in df.columns:
    d = cc[k] - base[k]
    better = (d < 0) if k in LOWER else (d > 0)
    delta[k] = f"{d:+.4f}  {'better' if better else 'worse'}"
df.loc['delta (CC - baseline)'] = pd.Series(delta)
df

In [ ]:
import matplotlib.pyplot as plt

metrics = [m for m in ['psnr', 'ssim', 'lpips', 'color_error', 'fid'] if m in base]
fig, axes = plt.subplots(1, len(metrics), figsize=(3.2 * len(metrics), 3.4))
for ax, m in zip(axes, metrics):
    vals = [base[m], cc[m]]
    ax.bar(['base', 'CC'], vals, color=['#888', '#2a7ab0'])
    ax.set_title(f"{m}{' (lower better)' if m in LOWER else ''}", fontsize=9)
    for i, v in enumerate(vals):
        ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.margins(y=0.18)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side samples: LR input / SR output / ground truth, baseline vs CC.
import glob
import matplotlib.pyplot as plt
from PIL import Image


def gen_dir(exp):
    return sorted(glob.glob(f'checkpoints/{exp}/results_*'))[-1]


b, c = gen_dir('diffsr_celebA_small_baseline'), gen_dir('diffsr_celebA_small_cc')
names = [p.split('/')[-1] for p in sorted(glob.glob(f'{b}/SR/*.png'))[:5]]
fig, axes = plt.subplots(4, len(names), figsize=(2.1 * len(names), 8.6))
for j, n in enumerate(names):
    for i, (d, sub, lab) in enumerate([(b, 'LR', 'LR input'), (b, 'SR', 'baseline SR'),
                                       (c, 'SR', 'CC-ResDiff SR'), (b, 'HR', 'ground truth')]):
        axes[i, j].imshow(Image.open(f'{d}/{sub}/{n}'))
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(lab, loc='left', fontsize=9)
plt.tight_layout()
plt.show()

## 8. (Optional) How much does the GCC term actually steer training?

Loss *values* make the auxiliary term look negligible (~0.06% of the DDPM loss at lambda=0.1); its *gradient* is ~34% of the DDPM gradient, and the contribution is wildly uneven across timesteps (~0.002% at t=0 vs ~236% at t=T-1). Needs a trained checkpoint, so run this after section 5.

In [ ]:
!PYTHONPATH=. python -u tasks/analyze_gcc_gradients.py \
  --config configs/diffsr_celeb_small_cc.yaml --exp_name diffsr_celebA_small_baseline

## 9. (Optional) Ablations

Two 1-D sweeps at a reduced budget (5000 steps each, ~7 runs): `lambda_color` in {0.01, 0.1, 0.5, 1.0} at pool size 8, and `color_pool_size` in {4, 8, 16} at lambda 0.1, plus a no-GCC reference at the same budget. Every run gets an identical budget so the sweep does not partly measure training length. Allow ~1-1.5 hours; `--skip_existing` resumes.

In [ ]:
!PYTHONPATH=. python -u tasks/run_ablations.py --dry_run

In [ ]:
# !PYTHONPATH=. python -u tasks/run_ablations.py \
#     --rrdb_ckpt checkpoints/rrdb_celebA_small --skip_existing

In [ ]:
# import pandas as pd
# pd.read_csv('checkpoints/ablation_results/ablations.csv')

## 10. (Optional) DIV2K as a second dataset

Matched to the CelebA setup (4x, 16x16 -> 64x64 patches, same capacity and schedule). Expects `data/raw/DIV2K_train_HR/*.png`; if no `DIV2K_valid_HR` is present a deterministic tail of the train split is held out as test.

Note DIV2K validation uses 96 tiles rather than 24: with 24, validation PSNR swung 2.2 dB between checks while SSIM and LPIPS improved monotonically, and that noise was driving checkpoint selection.

Adds ~2.5 hours.

In [ ]:
# !wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip -P data/raw
# !unzip -q data/raw/DIV2K_train_HR.zip -d data/raw && rm data/raw/DIV2K_train_HR.zip
# !PYTHONPATH=. python -u data_gen/df2k.py --config configs/df2k4x_small.yaml

In [ ]:
# !PYTHONPATH=. python -u tasks/trainer.py --config configs/rrdb/df2k4x_pretrain_small.yaml \
#     --exp_name rrdb_div2k_small --reset --hparams="num_workers={NUM_WORKERS}"
# !PYTHONPATH=. python -u tasks/trainer.py --config configs/diffsr_df2k4x_small.yaml \
#     --exp_name diffsr_div2k_small_baseline --reset --hparams="rrdb_ckpt=checkpoints/rrdb_div2k_small,num_workers={NUM_WORKERS}"
# !PYTHONPATH=. python -u tasks/trainer.py --config configs/diffsr_df2k4x_small_cc.yaml \
#     --exp_name diffsr_div2k_small_cc --reset --hparams="rrdb_ckpt=checkpoints/rrdb_div2k_small,num_workers={NUM_WORKERS}"
# !PYTHONPATH=. python -u tasks/evaluate.py --config configs/diffsr_df2k4x_small.yaml --exp_name diffsr_div2k_small_baseline
# !PYTHONPATH=. python -u tasks/evaluate.py --config configs/diffsr_df2k4x_small_cc.yaml --exp_name diffsr_div2k_small_cc

---
### Reading the results

On a local RTX 3050 run of this exact configuration, CC-ResDiff reduced `color_error` by **70.6%** and improved LR-PSNR by **0.81 dB**, at a cost of **0.12 dB** PSNR and a **+3.8 FID** regression; LPIPS was unchanged within noise. Expect the same direction, not identical numbers — a different training seed changes the weights.

Two caveats worth carrying into the writeup:

* **Single seed.** Evaluation is deterministic given a seed, but *training* seed variance is not measured here. The large `color_error` effect should replicate; the small PSNR and FID deltas may not.
* **FID at n=300** is heavily biased upward and only meaningful as a relative comparison between arms on the same images, never as an absolute number.